# 3 · Train and predict

Train on the image/label pairs saved in notebook 2, then predict land cover for the KHDTK forest from notebook 2.

The flow is: **load tiles → train → predict chips → save maps**.

**Data and interpretation.** These are expert-annotated training labels from [Tait et al. (2021)](https://doi.org/10.1594/PANGAEA.933475), paired with Sentinel-2 L2A imagery in the [teaching subset](https://huggingface.co/datasets/fatmur/dynamicworld-300). They have ten classes, including cloud, and nodata 255. They are not the automated nine-class Dynamic World probability product.

Keep tiles in their original train/validation/test splits before making chips. Nearby or overlapping tiles can still leak spatial information: inspect footprints before claiming geographic generalization. These short runs are workflow checks, not evidence of mapping accuracy.


**Run:** use the repository `.venv` kernel with `examples/` as the working directory. Requires `uv sync`, internet access, and the data paths described below.

In [ ]:
import warnings
from pathlib import Path

import kornia.augmentation as K
import lightning as L
import torch
import xarray as xr
from huggingface_hub import snapshot_download
from torch.utils.data import DataLoader, Dataset

import geosave as gs

L.seed_everything(0)
warnings.filterwarnings("ignore", ".*does not have many workers.*")

root = Path(
    snapshot_download(
        "fatmur/dynamicworld-300",
        repo_type="dataset",
        revision="5e65026ad47e1f02f9a48a3efee1028db3bb0d1a",
        local_dir="data/dynamic_world",
    )
)

## Load the classes

Use the dataset's class names, colours and ignore index throughout this notebook.

In [ ]:
import yaml

classes = yaml.safe_load((root / "classes.yaml").read_text())
IGNORE_INDEX = classes["ignore_index"]
CLASS_MAP = classes["class_map"]
COLOR_MAP = classes["color_map"]
NUM_CLASSES = len(CLASS_MAP)
CLASS_MAP

## Open the prepared pairs

Notebook 2 saved image and label files on the same grid, with fixed splits. Group their catalog rows by day and reopen each pair. Keep acquisition and preparation outside the training Dataset.

In [ ]:
def load_pairs(rows):
    groups = rows.gs.sample(collections=["sentinel-2-l2a", "label"], freq="D")
    images = []
    labels = []

    for rows in groups:
        with rows.gs.to_datatree() as tree:
            image = tree["sentinel-2-l2a"].gs.to_dataset().squeeze(drop=True)
            label = tree["label"].gs.to_dataset()["label"].squeeze(drop=True)

            images.append(image.load())
            labels.append(label.load())

    return images, labels


catalog = gs.open_vector("data/my_dataset/catalog.parquet")

train_images, train_labels = load_pairs(catalog[catalog["split"] == "train"])
val_images, val_labels = load_pairs(catalog[catalog["split"] == "val"])
test_images, test_labels = load_pairs(catalog[catalog["split"] == "test"])

In [ ]:
train_images[0].gs.plot(xlabel="") + train_labels[0].gs.plot(xlabel="")

## Prepare the acquired imagery

Unlike the complete teaching tiles used in notebook 0, these newly acquired scenes can have missing coverage. Decode reflectance and inspect it before training. Where an image has no observation, use zero as its input and 255 as its target so the loss ignores that location. Existing label IDs are otherwise unchanged.

Do this once, before creating chips or a training Dataset. The small collection fits in memory; for a larger project, prepare and save it in batches.

In [ ]:
bands = ["B02", "B03", "B04", "B08"]


def prepare_pairs(images, labels, *, bands, ignore_index):
    prepared_images = []
    prepared_labels = []

    for image, label in zip(images, labels, strict=True):
        reflectance = image[bands].gs.mask_and_scale()
        observed = reflectance.to_array().notnull().all("variable")

        prepared_images.append(reflectance.fillna(0))
        prepared_labels.append(label.where(observed, ignore_index))

    return prepared_images, prepared_labels


train_images, train_labels = prepare_pairs(
    train_images, train_labels, bands=bands, ignore_index=IGNORE_INDEX
)
val_images, val_labels = prepare_pairs(
    val_images, val_labels, bands=bands, ignore_index=IGNORE_INDEX
)
test_images, test_labels = prepare_pairs(
    test_images, test_labels, bands=bands, ignore_index=IGNORE_INDEX
)

## Cut matching image and label chips

The model takes 128 × 128 pixels. Cut image and label chips using the same size and order. At a scene's edge, pad images with zero and labels with 255. Padding is ignored by the loss and metrics.

Use no overlap for training and evaluation: every original validation or test pixel is counted once. We will use overlap later when blending predictions into a forest map. `Chips` reads a window when indexed; it does not load every chip at construction.

In [ ]:
def cut_pairs(images, labels, *, size, ignore_index):
    image_chips = gs.Chips(images, size=size, constant_value=0)
    label_chips = gs.Chips(
        labels, size=size, constant_value=ignore_index
    )

    return image_chips, label_chips


train_images, train_labels = cut_pairs(
    train_images, train_labels, size=128, ignore_index=IGNORE_INDEX
)
val_images, val_labels = cut_pairs(
    val_images, val_labels, size=128, ignore_index=IGNORE_INDEX
)
test_images, test_labels = cut_pairs(
    test_images, test_labels, size=128, ignore_index=IGNORE_INDEX
)

## Return tensors for training

The Dataset only selects a prepared chip pair and converts its arrays to tensors. It has no preparation, inference, or map-merging mode.

In [ ]:
class DynamicWorld(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index].gs.to_tensor(dtype="float32")
        label = self.labels[index].gs.to_tensor(dtype="int64")

        return image, label


train = DynamicWorld(train_images, train_labels)
val = DynamicWorld(val_images, val_labels)
test = DynamicWorld(test_images, test_labels)

train_loader = DataLoader(train, batch_size=8, shuffle=True)
val_loader = DataLoader(val, batch_size=8)
test_loader = DataLoader(test, batch_size=8)

## Build the model

Keep the Lightning training loop from notebook 1. Apply the same flips to each image and label during training. Validation and test use the unaugmented chips and ignore padded or unlabelled pixels.

In [ ]:
from torch import nn
from geosave.model import build_model
from torchmetrics import MetricCollection
from torchmetrics.classification import MulticlassAccuracy, MulticlassJaccardIndex


class Segmenter(L.LightningModule):
    def __init__(
        self, num_classes, ignore_index, in_channels=4, input_size=128, width=16
    ):
        super().__init__()

        self.model = build_model(
            {
                "encoder": {
                    "name": "small_cnn",
                    "init_args": {
                        "in_channels": in_channels,
                        "input_size": input_size,
                        "width": width,
                    },
                },
                "decoder": {"name": "unet"},
                "head": {
                    "name": "segmentation",
                    "init_args": {"num_classes": num_classes},
                },
            }
        )

        self.input_size = input_size
        self.augment = K.AugmentationSequential(
            K.RandomHorizontalFlip(),
            K.RandomVerticalFlip(),
            data_keys=["image", "mask"],
            keepdim=True,
        )

        self.val_metrics = MetricCollection(
            {
                "miou": MulticlassJaccardIndex(
                    num_classes=num_classes, ignore_index=ignore_index
                ),
                "accuracy": MulticlassAccuracy(
                    num_classes=num_classes, ignore_index=ignore_index, average="micro"
                ),
            },
            prefix="val_",
        )
        self.test_metrics = self.val_metrics.clone(prefix="test_")

        self.criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)

    def forward(self, image):
        return self.model(image=image)

    def training_step(self, batch, batch_index):
        image, label = self.augment(*batch)
        logits = self(image)
        loss = self.criterion(logits, label)

        self.log("train_loss", loss)

        return loss

    def validation_step(self, batch, batch_index):
        image, label = batch
        logits = self(image)

        self.val_metrics(logits, label)
        self.log_dict(self.val_metrics)

    def test_step(self, batch, batch_index):
        image, label = batch
        logits = self(image)

        self.test_metrics(logits, label)
        self.log_dict(self.test_metrics)

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=1e-2)


module = Segmenter(
    num_classes=NUM_CLASSES,
    ignore_index=IGNORE_INDEX,
    in_channels=4,
    input_size=128,
    width=16,
)
module

## Train and evaluate

Run six short epochs, using six training batches per epoch. Validation covers all chips from the two validation scenes. Logs and checkpoints go to `lightning_logs/03_train/`; rerunning starts a fresh model.

In [ ]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import TensorBoardLogger

logger = TensorBoardLogger("lightning_logs", name="03_train", version=0)
checkpoint = ModelCheckpoint(save_last=True)

trainer = L.Trainer(
    max_epochs=6,
    limit_train_batches=6,
    log_every_n_steps=1,
    logger=logger,
    callbacks=[checkpoint],
)
trainer.fit(module, train_loader, val_loader)

Evaluate on the held-out test split.

In [ ]:
trainer.test(module, test_loader)

### Optional: inspect training logs

Run this in a separate cell when you want TensorBoard. It starts a local service.

```python
%load_ext tensorboard
%tensorboard --logdir lightning_logs
```

## Open the forest observations

The forest has no reference labels. Open one image per date from notebook 2's catalog. Keep the observation mask for the final maps, and fill missing image values before prediction.

In [ ]:
region = gs.open_vector("khdtk.geojson")
items = gs.open_vector("data/khdtk/catalog.parquet")
groups = items.gs.sample(collections=["sentinel-2-l2a"], freq="D")

images = []
masks = []
dates = []
for rows in groups:
    with rows.gs.to_datatree() as tree:
        scene = tree["sentinel-2-l2a"].gs.to_dataset().squeeze(drop=True)
        reflectance = scene[bands].gs.mask_and_scale().load()

        masks.append(reflectance.to_array().notnull().all("variable"))
        images.append(reflectance.fillna(0))
        dates.append(tree.gs.region.timespan[0])

## Predict overlapping chips

Overlap lets us blend neighboring predictions. Keep each chip's key with its scores so `merge` can put them back in the right place. This loop calls the model directly; it does not need a training Dataset.

In [ ]:
chips = gs.Chips(images, size=128, overlap=64)

module.eval()
predictions = []
with torch.inference_mode():
    for key in chips:
        image = chips[key].gs.to_tensor(dtype="float32")
        batch = image.unsqueeze(0).to(module.device)
        scores = module(batch)[0].cpu().numpy()

        predictions.append((key, scores))

merged = chips.merge(predictions, window="hamming")

## Put the scores back on the map

`merge` returns one array per original image. Restore each image's grid and date, choose the highest-scoring class, and leave missing observations as 255. Attach the class names and colors for plotting and saving.

In [ ]:
maps = []
for image, pixels, observed, date in zip(images, merged, masks, dates, strict=True):
    scores = xr.DataArray(
        pixels,
        dims=("class_id", "y", "x"),
        coords={"class_id": sorted(CLASS_MAP)},
    )
    scores = scores.gs.write_grid(image.gs.geobox)

    cover = scores.idxmax("class_id")
    cover = cover.where(observed, IGNORE_INDEX).astype("uint8")
    cover = cover.assign_coords(time=date)
    maps.append(cover)

forest_cover = xr.concat(
    maps, dim="time", join="exact", combine_attrs=gs.attrs.combine_attrs
).rename("label")
forest_cover = forest_cover.gs.write_nodata(IGNORE_INDEX)
forest_cover = forest_cover.gs.write_legend(CLASS_MAP, COLOR_MAP)
forest_cover.gs.plot(xlabel="")

## Save the maps

Write one TIFF per date and a catalog pointing to those files. Keep the catalog and raster folder together when moving the results.

In [ ]:
import pystac

collection = pystac.Collection(
    id="land_cover",
    description="Predicted forest land cover",
    extent=pystac.Extent.from_items(items.gs.to_items()),
)

folder = Path("data/khdtk_pred")

cover_catalog = forest_cover.gs.to_catalog(
    folder,
    format_paths="land_cover/{time:%Y%m%d}.tif",
    collection=collection,
    split_asset=(),
    format_id="{time:%Y%m%d}",
    overwrite=True,
)
cover_catalog.gs.to_geoparquet(folder / "catalog.parquet", overwrite=True)
cover_catalog[["id", "collection", "datetime"]]

In [ ]:
forest_cover.gs.explore(basemap="Imagery") * region

## Keep this trained model for the next observation

A Lightning checkpoint keeps the optimizer and training state for resuming a run. For the next lesson, also save this model's architecture and trained weights, along with the class definitions. We will reopen these files in a fresh kernel and recover exactly the same model.

Rerunning this cell replaces the handoff in `data/landcover/`. Keep a separate folder for each experiment you intend to compare.

In [ ]:
handoff = Path("data/landcover")
handoff.mkdir(parents=True, exist_ok=True)

(handoff / "model.yaml").write_text(
    yaml.safe_dump(module.model.stage_specs, sort_keys=False)
)
torch.save(module.model.state_dict(), handoff / "weights.pt")
(handoff / "classes.yaml").write_text(yaml.safe_dump(classes, sort_keys=False))

## More things you can do with the forest maps (optional)

The maps and trained model are saved. Continue to [notebook 4](04_recipe.ipynb), or try these review tasks.

### Give the forest manager a class summary
Count only predictions inside the project boundary. Show the modeled class distribution beside the map, and keep missing pixels out of the denominator.

In [ ]:
inside = region.gs.rasterize(forest_cover)
forest_cover.where(inside, IGNORE_INDEX).gs.class_percent()

### Review suspected changes with the original imagery
A drop in predicted tree cover is a reason to inspect the two dates side by side. Clouds, shadows, season, and model error can all change the prediction. Confirm a suspected change with independent observations before calling it forest loss.

### Prepare a GIS review layer
Vectorize one dated class map with `.gs.vectorize(value_name="class_id")`, join the class names, and let reviewers annotate disputed areas. Preserve the date and model run with the layer.

**Next:** [keep the model's input recipe](04_recipe.ipynb), so mapping another observation does not accidentally change band order, reflectance scaling, or missing-pixel handling.